In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [2]:
meta_csv_path = 'B0531+21_59000_48386_channels_meta.csv'
channels_dataset_path = 'B0531+21_59000_48386_channels.npy'
meta = pd.read_csv(meta_csv_path)
meta = meta.fillna("None")
dataset = np.load(channels_dataset_path)

/tmp/ipykernel_159347/1975792744.py:3: DtypeWarning: Columns (21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(meta_csv_path)


In [15]:
# Проверка на соответствие размеров
assert len(meta) == len(dataset), "Длина meta и dataset не совпадает!"

In [14]:
meta.head(257)

,mean_o,std_o,skew_o,kurt_o,mean_n,std_n,skew_n,kurt_n,mean_o_ratio,std_o_ratio,skew_o_ratio,kurt_o_ratio,mean_n_ratio,std_n_ratio,skew_n_ratio,kurt_n_ratio,label
0,128.359375,21.264052,0.114631,-0.140737,122.070175,47.750854,0.114631,-0.140737,1.007947,1.002565,0.701702,-3.980736,1.019578,1.006820,0.676858,1.575076,None
1,128.261719,20.569192,0.199155,-0.028248,146.817460,41.791374,0.199155,-0.028248,1.007180,0.969804,1.219106,-0.799001,1.226277,0.881165,1.175943,0.316145,None
2,127.324219,21.419552,0.365953,0.379996,117.855932,46.469537,0.365953,0.379996,0.999819,1.009897,2.240143,10.748126,0.984379,0.979803,2.160830,-4.252761,None
3,127.824219,23.605902,-0.004858,-0.065768,123.508065,48.734766,-0.004858,-0.065768,1.003745,1.112980,-0.029736,-1.860230,1.031588,1.027565,-0.028684,0.736046,None
4,130.824219,21.604496,-0.174239,-0.384922,131.054054,49.826586,-0.174239,-0.384922,1.027302,1.018617,-1.066585,-10.887461,1.094615,1.050586,-1.028822,4.307892,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252,127.953125,21.674207,0.322218,0.339468,123.633333,46.238308,0.322218,0.339468,1.004757,1.021904,1.972423,9.601803,1.032634,0.974928,1.902589,-3.799190,None
253,127.187500,22.243942,0.126423,0.098516,123.410853,44.143017,0.126423,0.098516,0.998745,1.048766,0.773885,2.786523,1.030776,0.930749,0.746486,-1.102557,None
254,128.812500,21.022960,-0.070946,-0.288952,119.339450,49.375025,-0.070946,-0.288952,1.011505,0.991198,-0.434292,-8.172951,0.996770,1.041065,-0.418915,3.233829,None
255,128.488281,21.262126,0.151461,-0.040589,107.496000,43.544833,0.151461,-0.040589,1.008959,1.002475,0.927155,-1.148054,0.897849,0.918136,0.894329,0.454256,None


In [13]:
meta = meta[['mean_o', 'std_o', 'skew_o', 'kurt_o', 'mean_n', 'std_n',
       'skew_n', 'kurt_n', 'mean_o_ratio', 'std_o_ratio', 'skew_o_ratio',
       'kurt_o_ratio', 'mean_n_ratio', 'std_n_ratio', 'skew_n_ratio',
       'kurt_n_ratio', 'label']]

In [16]:
def make_balanced_subset(meta: pd.DataFrame,
                         dataset: np.ndarray,
                         label_col: str = 'label',
                         labels_to_sample=('NBRFI', 'None'),
                         n_per_class: int = 10_000,
                         random_state: int | None = 42):
    """
    Создаёт сбалансированный поднабор по указанным классам.
    
    labels_to_sample: какие лейблы брать (по умолчанию 'NBRFI' и 'None')
    n_per_class: сколько элементов на каждый класс (максимум, если меньше — возьмёт сколько есть)
    """
    rng = np.random.default_rng(random_state)

    all_indices = []

    for label in labels_to_sample:
        # Индексы строк с данным лейблом
        label_indices = meta.index[meta[label_col] == label].to_numpy()

        if len(label_indices) == 0:
            raise ValueError(f"В meta нет ни одной строки с label='{label}'")

        # Сколько реально можем взять
        k = min(n_per_class, len(label_indices))
        if k < n_per_class:
            print(f"Предупреждение: для класса '{label}' доступно только {len(label_indices)} объектов, "
                  f"будут использованы все {k}.")

        # Сэмплируем без повторений
        chosen = rng.choice(label_indices, size=k, replace=False)
        all_indices.append(chosen)

    # Объединяем индексы всех классов
    all_indices = np.concatenate(all_indices)

    # Перемешиваем итоговый набор, чтобы классы перемешались
    rng.shuffle(all_indices)

    # Делаем поднаборы
    meta_subset = meta.iloc[all_indices].reset_index(drop=True)
    dataset_subset = dataset[all_indices]

    return meta_subset, dataset_subset, all_indices

In [18]:
# Пример использования:
meta_subset, dataset_subset, used_indices = make_balanced_subset(
    meta,
    dataset,
    label_col='label',
    labels_to_sample=('NBRFI', 'None'),
    n_per_class=10_000,
    random_state=42
)

# Сохраняем результаты
meta_subset.to_csv('B0531+21_59000_48386_subset_channels_meta.csv', index=False)
np.save('B0531+21_59000_48386_subset_channels.npy', dataset_subset)
np.save('B0531+21_59000_48386_subset_indices.npy', used_indices)

In [21]:

# y: 1 для NBRFI, 0 для None (по subset meta)
y = (meta_subset['label'].values == 'NBRFI').astype(np.int64)

idx = np.arange(len(meta_subset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    idx, y, test_size=0.2, random_state=42, stratify=y
)

val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

np.savez(
    "split_indices.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx
)

# опционально сохраняем y
np.save("subset_y.npy", y)